# Pepare VneuroTK Data 
> 以 [NOD-MEG](https://openneuro.org/datasets/ds005810) 为例，演示从读取 MEG 数据、
> 配置试次结构、提取 DNN 特征到持久化保存的完整流程。


**工作流**：
1. 读取 MEG 数据
2. 配置trial结构
3. 获取不同Neuro数据视图（epochs / continuous）
4. 加载视觉模型，选取目标层
5. 整合特征提取（`data.vision.extract_from()`）
6. 访问与索引视觉特征
7. 保存与重加载（HDF5 + LazyH5Dict）


In [1]:
import os
import tempfile
from pathlib import Path

import mne  # type: ignore
import numpy as np
import pandas as pd
import torch  # type: ignore

import vneurotk as vtk
from vneurotk.datasets import sample
from vneurotk.io import MNEPath, VTKPath

os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"  # 可选：国内镜像加速下载

root = sample.data_path("nod-meg")
nod = root / "nod-meg"
SAVE_ROOT = Path(tempfile.mkdtemp()) / "mne" / "NOD-MEG"
SAVE_ROOT.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

## 1. Read data

`vtk.read()` uses lazy loading, pass `pre_load=True` to load immediately.

In [2]:
sub, session, run = sample.NOD_SUBJECT, sample.NOD_SESSION, sample.NOD_RUN

mne_path = MNEPath(
    root=nod / "meg",
    subject=sub,
    session=session,
    task=sample.NOD_TASK,
    run=run,
    suffix="meg_clean",
    extension=".fif",
)

data: vtk.BaseData = vtk.read(mne_path)
data

Loading MNE metadata from /var/tmp/zzl-zgh/cache/vneurotk/vneurotk-samples/nod-meg/meg/sub-01_ses-ImageNet01_task-ImageNet_run-01_meg_clean.fif
MNE-like metadata loaded: 80000 timepoints, 273 channels, sfreq=250.0 Hz


Time points,80000
Channels,273
Sampling frequency,250.00 Hz
Highpass,0.10 Hz
Lowpass,100.00 Hz
Data mode,continuous
Status,Not configured
Status,Not configured


# 2. Configure trial structure


`configure()` binds stimulus IDs, onset sample indices, a time window,
and an image database to the recording. After this call `data.neuro`
can be accessed as `epochs` or `continuous`.

In [3]:
submeta = pd.read_csv(nod / "events" / f"sub-{sub}_events.csv")
runmeta = submeta.query(f"run == {int(run)} and session == '{session}'")
stim_ids = runmeta["image_id"].values
stims = {sid: nod / "stimuli" / f"{sid}.JPEG" for sid in np.unique(stim_ids)}

raw = mne.io.read_raw(mne_path.fpath, preload=False, verbose=False)
vision_onsets = vtk.utils.get_event_samples(raw, event_name="stim_on")

data.configure(
    vision_onsets=vision_onsets,
    stim_ids=stim_ids,
    vision_db=stims,
    trial_window=[-0.2, 0.8],
)
data

Configured (raw): 200 trials, 200 unique stimuli


Time points,80000
Channels,273
Sampling frequency,250.00 Hz
Highpass,0.10 Hz
Lowpass,100.00 Hz
Data mode,continuous
n_visual,200
Baseline,"[-50, 0]"
Trial window,"[-50, 200]"


# 3. Get different views of neural data
---------------

`data.neuro` triggers the lazy load and returns a `NeuroData`
(`np.ndarray` subclass). Two views are available:

- **`.epochs`** — `(n_trials, n_times, nchan)`
- **`.continuous`** — `(total_samples, nchan)`

In [4]:
print(f"n_trials  = {data.n_trials}")
print(f"nchan     = {data.nchan}")
print(f"data_mode = {data.data_mode}")

neuro = data.neuro
print(f"\nraw shape  : {neuro.shape}")
print(f"epochs     : {data.neuro.epochs.shape}")
print(f"continuous : {data.neuro.continuous.shape}")

Lazy-loading neuro data...
Reading MNE data into memory from /var/tmp/zzl-zgh/cache/vneurotk/vneurotk-samples/nod-meg/meg/sub-01_ses-ImageNet01_task-ImageNet_run-01_meg_clean.fif


n_trials  = 200
nchan     = 273
data_mode = continuous

raw shape  : (80000, 273)


WARNING | neuro is already in continuous format (shape (80000, 273)), returning as-is


epochs     : (200, 250, 273)
continuous : (80000, 273)


# 4. Load vision model
----------------------

 `VisionModel` is a pure activation extractor — no pooling or CLS-token
 extraction. Three backends are supported: `"transformers"`, `"timm"`,
 `"thingsvision"`.

 First, list any models already cached on this machine:

In [5]:
vtk.print_cached_models()

  model_id                                               size     last used  
  facebook/dinov2-base                                 330 MB    2026-05-05  
  facebook/dinov3-vitb16-pretrain-lvd1689m             327 MB    2026-05-05  
  facebook/dinov3-vits16-pretrain-lvd1689m              82 MB    2026-05-05  
  google/siglip-base-patch16-224                       778 MB    2026-05-05  
  google/siglip2-base-patch16-224                      1.4 GB    2026-05-05  
  microsoft/resnet-50                                   98 MB    2026-05-05  
  openai/clip-vit-base-patch32                         1.7 GB    2026-05-05  
  Qwen/Qwen3-Embedding-8B                             14.1 GB    2026-05-06  
  timm/resnet50.a1_in1k                                 98 MB    2026-05-05  
  timm/resnetv2_50.a1h_in1k                             98 MB    2026-05-05  
  timm/vit_base_patch16_224.augreg2_in21k_ft_in1k      330 MB    2026-05-05  
  timm/vit_base_patch16_224.dino                       327 MB    2026-05-05  
  ViT-B-32.pt                                          338 MB    2026-05-05

───────────────────────────────────────────────────────────────────────────────────────────────────────────────────

██ transformers   ██ timm   ██ clip

In [6]:
# Load DINOv2-Base (downloads ~330 MB on first use):
model = vtk.VisionModel(
    "facebook/dinov2-base",
    backend="transformers",
    device=DEVICE,
)
model.print_modules(max_depth=3)

Loading transformers model: facebook/dinov2-base (pretrained=True)


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

Loaded transformers model: facebook/dinov2-base
VisionModel ready | model=facebook/dinov2-base | backend=transformers | modules=12


 Module  (⊡ container   • leaf)                              Type                             Param 
 ⊡ ├─ embeddings                                             Dinov2Embeddings       ░░░░░░░░  1.51M 
 ⊡ │  ├─ patch_embeddings                                    Dinov2PatchEmbeddings  ░░░░░░░░ 452.4K 
 • │  │  └─ projection  weight:[768, 3, 14, 14]  bias:[768]  Conv2d                 ░░░░░░░░ 452.4K 
 • │  └─ dropout                                             Dropout                ░░░░░░░░      - 
 ⊡ ├─ encoder                                                Dinov2Encoder          ████████ 85.07M 
 ⊡ │  └─ layer                                               ModuleList             ████████ 85.07M 
 ⊡ │     ├─ 0                                                Dinov2Layer            ░░░░░░░░  7.09M 
 ⊡ │     ├─ 1                                                Dinov2Layer            ░░░░░░░░  7.09M 
 ⊡ │     ├─ 2                                                Dinov2Layer            ░░░░░░░░  7.09M 
 ⊡ │     ├─ 3                                                Dinov2Layer            ░░░░░░░░  7.09M 
 ⊡ │     ├─ 4                                                Dinov2Layer            ░░░░░░░░  7.09M 
 ⊡ │     ├─ 5                                                Dinov2Layer            ░░░░░░░░  7.09M 
 ⊡ │     ├─ 6                                                Dinov2Layer            ░░░░░░░░  7.09M 
 ⊡ │     ├─ 7                                                Dinov2Layer            ░░░░░░░░  7.09M 
 ⊡ │     ├─ 8                                                Dinov2Layer            ░░░░░░░░  7.09M 
 ⊡ │     ├─ 9                                                Dinov2Layer            ░░░░░░░░  7.09M 
 ⊡ │     ├─ 10                                               Dinov2Layer            ░░░░░░░░  7.09M 
 ⊡ │     └─ 11                                               Dinov2Layer            ░░░░░░░░  7.09M 
 • └─ layernorm  weight:[768]  bias:[768]                    LayerNorm              ░░░░░░░░   1.5K

Type legend:

██ Dinov2Embeddings   ██ Dinov2PatchEmbeddings   ██ Conv2d   ██ Dropout   ██ Dinov2Encoder   ██ ModuleList   ██ 
Dinov2Layer   ██ LayerNorm

In [7]:
# Select target layers by module type and register them:

all_modules = model.module_list
dinov2_layers = [m for m in all_modules if m.module_type == "Dinov2Layer"]
last_norm = [m for m in all_modules if m.module_type == "LayerNorm"][-1]
target_layers = dinov2_layers + [last_norm]
vtk.print_modules(target_layers)

 Module  (⊡ container   • leaf)                    Type                            Param 
 ⊡ │     ├─ 0                                      Dinov2Layer           ████████  7.09M 
 ⊡ │     ├─ 1                                      Dinov2Layer           ████████  7.09M 
 ⊡ │     ├─ 2                                      Dinov2Layer           ████████  7.09M 
 ⊡ │     ├─ 3                                      Dinov2Layer           ████████  7.09M 
 ⊡ │     ├─ 4                                      Dinov2Layer           ████████  7.09M 
 ⊡ │     ├─ 5                                      Dinov2Layer           ████████  7.09M 
 ⊡ │     ├─ 6                                      Dinov2Layer           ████████  7.09M 
 ⊡ │     ├─ 7                                      Dinov2Layer           ████████  7.09M 
 ⊡ │     ├─ 8                                      Dinov2Layer           ████████  7.09M 
 ⊡ │     ├─ 9                                      Dinov2Layer           ████████  7.09M 
 ⊡ │     ├─ 10                                     Dinov2Layer           ████████  7.09M 
 ⊡ │     └─ 11                                     Dinov2Layer           ████████  7.09M 
 • └─ layernorm  weight:[768]  bias:[768]          LayerNorm             ░░░░░░░░   1.5K

Type legend:

██ Dinov2Layer   ██ LayerNorm

In [8]:
model.set_selector(target_layers)

Selector updated | modules=13


You can also select by type or name string — the result is identical::

    model.set_selector(module_type=["Dinov2Layer"], module_name=["layernorm"])

# 5. Standalone feature extraction — model.extract()
---------------------------------------------------

`model.extract()` operates on any `{stim_id: image}` mapping and
returns a `VisualRepresentations` object — independent of any `BaseData`.

In [9]:
demo_imgs = {sid: nod / "stimuli" / f"{sid}.JPEG" for sid in list(np.unique(stim_ids))[:20]}
vrs = model.extract(demo_imgs, batch_size=16)
print(vrs)

VisionModel:   0%|          | 0/2 [00:00<?, ?batch/s]

Extracted | n=20 | batch_size=16 | modules=13


VisualRepresentations(20 stimuli x 13 modules, models=['facebook/dinov2-base'])


Each `(model, module)` combination has associated metadata:

In [10]:
vrs.meta

,model,module_type,module_name,shape
0,facebook/dinov2-base,Dinov2Layer,encoder.layer.0,"(20, 257, 768)"
1,facebook/dinov2-base,Dinov2Layer,encoder.layer.1,"(20, 257, 768)"
2,facebook/dinov2-base,Dinov2Layer,encoder.layer.2,"(20, 257, 768)"
3,facebook/dinov2-base,Dinov2Layer,encoder.layer.3,"(20, 257, 768)"
4,facebook/dinov2-base,Dinov2Layer,encoder.layer.4,"(20, 257, 768)"
5,facebook/dinov2-base,Dinov2Layer,encoder.layer.5,"(20, 257, 768)"
6,facebook/dinov2-base,Dinov2Layer,encoder.layer.6,"(20, 257, 768)"
7,facebook/dinov2-base,Dinov2Layer,encoder.layer.7,"(20, 257, 768)"
8,facebook/dinov2-base,Dinov2Layer,encoder.layer.8,"(20, 257, 768)"
9,facebook/dinov2-base,Dinov2Layer,encoder.layer.9,"(20, 257, 768)"


In [11]:
# `VisualRepresentations` supports multiple indexing styles:

layer_name = model.module_names[-1]
vr = vrs[layer_name]
print(f"str index  → VisualRepresentation   n_stim={vr.n_stim}, shape={vr.shape}")

arr = vrs.numpy(layer_name)
print(f"numpy()    → ndarray                 shape={arr.shape}")

# Boolean mask returns a subset:
subset = vrs[vrs.meta["module_type"] == "Dinov2Layer"]
print(f"bool mask  → VisualRepresentations   {len(subset)} layers")

# Select a subset of stimuli (all layers aligned):
vrs_5 = vrs.select(vrs.stim_ids[:5])
print(f"select(5)  → n_stim={vrs_5.n_stim}")

str index  → VisualRepresentation   n_stim=20, shape=(20, 257, 768)
numpy()    → ndarray                 shape=(20, 257, 768)
bool mask  → VisualRepresentations   12 layers
select(5)  → n_stim=5


# 6. Integrated feature extraction — data.vision.extract_from()
-------------------------------------------------------------

Calling `extract_from()` on `VisionData` uses the image database
already bound by `configure()`. Features are stored at unique-stimulus
granularity internally; onset-aligned (trial-order) arrays are produced
only at read time.

In [12]:
data.vision.extract_from(model, batch_size=16)
data.vision

extract_from: 200 stimuli, 13 modules, batch_size=16


VisionModel:   0%|          | 0/13 [00:00<?, ?batch/s]

Extracted | n=200 | batch_size=16 | modules=13
extract_from done: 13 modules stored.


VisionData(200 stimuli x 13 modules)

7. Index and slice visual representations
-----------------------------------------

`data.vision` supports three indexing semantics:

   * - Index
     - Returns
   * - `vision["layer_name"]`
     - Onset-aligned ndarray, shape `(n_trials, ...)`
   * - `vision[int]`
     - The *i*-th VR, onset-aligned ndarray
   * - `vision[bool_mask]`, 1 match
     - Onset-aligned ndarray
   * - `vision[bool_mask]`, multiple matches
     - `VisualRepresentations`

In [13]:
print(data.vision.meta)

layer_name = model.module_names[-1]
feat = data.vision[layer_name]
print(f"vision[{layer_name!r}] → {feat.shape}  (n_trials={data.n_trials})")

feat0 = data.vision[0]
print(f"vision[0] → {feat0.shape}")

meta = data.vision.meta
dinov2_mask = meta["module_type"] == "Dinov2Layer"
layernorm_mask = meta["module_name"] == "layernorm"

print(f"single-match mask  → ndarray : {data.vision[layernorm_mask].shape}")

feat_multi = data.vision[dinov2_mask]
print(f"multi-match mask   → VisualRepresentations : {len(feat_multi)} layers")

# Further filter to a single layer:
filtered_mask = feat_multi.meta.query("module_name == 'encoder.layer.11'").index
filtered_vr = feat_multi[filtered_mask]
print(f"further filtered   → VisualRepresentation : {data.vision[filtered_vr].shape}")

                   model  module_type       module_name            shape
0   facebook/dinov2-base  Dinov2Layer   encoder.layer.0  (200, 257, 768)
1   facebook/dinov2-base  Dinov2Layer   encoder.layer.1  (200, 257, 768)
2   facebook/dinov2-base  Dinov2Layer   encoder.layer.2  (200, 257, 768)
3   facebook/dinov2-base  Dinov2Layer   encoder.layer.3  (200, 257, 768)
4   facebook/dinov2-base  Dinov2Layer   encoder.layer.4  (200, 257, 768)
5   facebook/dinov2-base  Dinov2Layer   encoder.layer.5  (200, 257, 768)
6   facebook/dinov2-base  Dinov2Layer   encoder.layer.6  (200, 257, 768)
7   facebook/dinov2-base  Dinov2Layer   encoder.layer.7  (200, 257, 768)
8   facebook/dinov2-base  Dinov2Layer   encoder.layer.8  (200, 257, 768)
9   facebook/dinov2-base  Dinov2Layer   encoder.layer.9  (200, 257, 768)
10  facebook/dinov2-base  Dinov2Layer  encoder.layer.10  (200, 257, 768)
11  facebook/dinov2-base  Dinov2Layer  encoder.layer.11  (200, 257, 768)
12  facebook/dinov2-base    LayerNorm         layer

# 8. Add activations from a second model (ResNet-50 via timm)
------------------------------------------------------------

In [18]:
model1 = vtk.VisionModel(
    "timm/resnet50.a1_in1k",
    backend="timm",
    device=DEVICE,
)
model1.set_selector(module_type="Bottleneck", module_name="global_pool")

data.vision.extract_from(model1, batch_size=16)
data.vision.meta

Loading timm model: timm/resnet50.a1_in1k (pretrained=True)
Loaded timm model: timm/resnet50.a1_in1k
VisionModel ready | model=timm/resnet50.a1_in1k | backend=timm | modules=16
Selector updated | modules=17
extract_from: all 17 modules already extracted for 'timm/resnet50.a1_in1k', skipping.


,model,module_type,module_name,shape
0,facebook/dinov2-base,Dinov2Layer,encoder.layer.0,"(200, 257, 768)"
1,facebook/dinov2-base,Dinov2Layer,encoder.layer.1,"(200, 257, 768)"
2,facebook/dinov2-base,Dinov2Layer,encoder.layer.2,"(200, 257, 768)"
3,facebook/dinov2-base,Dinov2Layer,encoder.layer.3,"(200, 257, 768)"
4,facebook/dinov2-base,Dinov2Layer,encoder.layer.4,"(200, 257, 768)"
5,facebook/dinov2-base,Dinov2Layer,encoder.layer.5,"(200, 257, 768)"
6,facebook/dinov2-base,Dinov2Layer,encoder.layer.6,"(200, 257, 768)"
7,facebook/dinov2-base,Dinov2Layer,encoder.layer.7,"(200, 257, 768)"
8,facebook/dinov2-base,Dinov2Layer,encoder.layer.8,"(200, 257, 768)"
9,facebook/dinov2-base,Dinov2Layer,encoder.layer.9,"(200, 257, 768)"


# 9. Save and reload
------------------

 `save()` writes neural data, all visual representations, trial
 configuration, and image pixels into a single HDF5 file.
 On reload, `vision.db` becomes a `LazyH5Dict` — images are decoded
 only when accessed.

In [22]:
save_path = VTKPath(SAVE_ROOT, subject=sub, session=session, task="ImageNet", run=run)
data.save(save_path)
loaded: vtk.BaseData = vtk.read(save_path)
loaded

Saved BaseData to /tmp/tmpddm6a52g/mne/NOD-MEG/sub-01_ses-ImageNet01_task-ImageNet_run-01.h5
Loading VTK data from /tmp/tmpddm6a52g/mne/NOD-MEG/sub-01_ses-ImageNet01_task-ImageNet_run-01.h5
Loaded VTK data (lazy): neuro shape (80000, 273)


Time points,80000
Channels,273
Sampling frequency,250.00 Hz
Highpass,0.10 Hz
Lowpass,100.00 Hz
Data mode,continuous
n_visual,200
Baseline,"[-50, 0]"
Trial window,"[-50, 200]"
